In [ ]:
!pip install transformers==4.41.2 accelerate librosa soundfile tqdm pandas

# HuBERT + Wav2Vec2 FEATURE EXTRACTION PIPELINE
# Uses preprocessed patient-only WAV files

In [ ]:
# Install packages
#!pip install transformers==4.41.2 accelerate librosa soundfile tqdm pandas


# Mount Google Drive
from google.colab import drive
from pathlib import Path
drive.mount("/content/drive", force_remount=True)

# define input and output folders
PROJECT = Path("/content/drive/MyDrive/asr project")
PREPROCESSED_ROOT = PROJECT / "preprocessed_patient_audio/Patients/"
FEATURE_ROOT = PROJECT / "features" / "ssl_patient_only"
assert PROJECT.exists(), f"Project folder not found: {PROJECT}"
assert PREPROCESSED_ROOT.exists(), f"Preprocessed folder not found: {PREPROCESSED_ROOT}"
FEATURE_ROOT.mkdir(parents=True, exist_ok=True)
print("Project:", PROJECT)
print("Preprocessed audio root:", PREPROCESSED_ROOT)
print("Feature output root:", FEATURE_ROOT)


# imports
import os
import gc
import numpy as np
import pandas as pd
import torch
import librosa
import transformers
from tqdm import tqdm
from transformers import (
    Wav2Vec2FeatureExtractor,
    Wav2Vec2ForSequenceClassification,
    HubertForSequenceClassification,
)
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)


# use gpu when available
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


# load audio


def load_wav_mono_16k(wav_path, target_sr=16000):
    """
    Load audio as mono 16 kHz torch tensor.
    Peak-normalized.
    """

    # load resample and convert audio to mono
    x, sr = librosa.load(
        str(wav_path),
        sr=target_sr,
        mono=True
    )

    # use float32 for the models
    x = x.astype(np.float32)

    # find the highest absolute amplitude
    max_abs = np.max(np.abs(x)) if len(x) > 0 else 0.0

    # normalize non silent audio
    if max_abs > 0:
        x = x / max_abs

    # convert the waveform to a torch tensor
    return torch.tensor(x, dtype=torch.float32)


def split_into_segments(
    waveform,
    sample_rate=16000,
    segment_seconds=10,
    min_segment_seconds=0.25,
):
    """
    Split waveform into fixed-length chunks.
    Last chunk is zero-padded.
    """

    # calculate segment and minimum lengths
    segment_len = int(sample_rate * segment_seconds)
    min_len = int(sample_rate * min_segment_seconds)
    segments = []

    # move through the waveform one segment at a time
    for start in range(0, waveform.numel(), segment_len):
        segment = waveform[start:start + segment_len]

        # skip very short final segments
        if segment.numel() < min_len:
            continue

        # pad shorter segments with zeros
        if segment.numel() < segment_len:
            pad_len = segment_len - segment.numel()
            segment = torch.nn.functional.pad(segment, (0, pad_len))

        segments.append(segment)

    # use one silent segment when no valid audio exists
    if len(segments) == 0:
        segments.append(torch.zeros(segment_len, dtype=torch.float32))

    return segments


# -----------------------------
# 6. Generic SSL extraction
# -----------------------------

def extract_ssl_selected_features(
    wav_path,
    feature_extractor,
    model,
    selected_layers=(6, -1),
    segment_seconds=10,
    sample_rate=16000,
    device="cpu",
):
    """
    Extract selected hidden-layer features from Wav2Vec2 or HuBERT.

    Returns dict:
        layer_6: np.ndarray shape [768]
        layer_last: np.ndarray shape [768]
        all_layers_mean: np.ndarray shape [768]
    """

    # load the audio waveform
    waveform = load_wav_mono_16k(
        wav_path,
        target_sr=sample_rate
    )

    # divide the audio into fixed segments
    segments = split_into_segments(
        waveform,
        sample_rate=sample_rate,
        segment_seconds=segment_seconds
    )

    # prepare the model for inference
    model.eval()
    model.to(device)

    # store embeddings by hidden layer
    per_layer_segment_embeddings = {}

    # disable gradient calculation
    with torch.no_grad():

        # process each audio segment
        for segment in segments:

            # prepare model inputs
            inputs = feature_extractor(
                segment.numpy(),
                sampling_rate=sample_rate,
                return_tensors="pt",
                padding=True,
            )

            # move all inputs to the selected device
            inputs = {
                k: v.to(device)
                for k, v in inputs.items()
            }

            # request hidden states from all layers
            outputs = model(
                **inputs,
                output_hidden_states=True,
            )

            hidden_states = outputs.hidden_states
            n_layers = len(hidden_states)

            # convert negative layer numbers to positive indices
            layer_indices = []

            for layer in selected_layers:
                if layer < 0:
                    layer_indices.append(n_layers + layer)
                else:
                    layer_indices.append(layer)

            # collect every layer for the all layer mean
            for layer_idx, hidden in enumerate(hidden_states):

                # average hidden frames into one vector
                emb = hidden.mean(dim=1).squeeze(0).detach().cpu().numpy()

                # create the layer list when needed
                if layer_idx not in per_layer_segment_embeddings:
                    per_layer_segment_embeddings[layer_idx] = []

                # save the segment embedding
                per_layer_segment_embeddings[layer_idx].append(emb)

    # average every layer across audio segments
    final_per_layer = {}

    for layer_idx, embeddings in per_layer_segment_embeddings.items():
        final_per_layer[layer_idx] = np.mean(
            np.stack(embeddings),
            axis=0
        ).astype(np.float32)

    n_layers = len(final_per_layer)

    # store the requested outputs
    result = {}

    for layer in selected_layers:

        # resolve the requested layer index and name
        if layer < 0:
            layer_idx = n_layers + layer
            layer_name = "last"
        else:
            layer_idx = layer
            layer_name = f"layer_{layer}"

        result[layer_name] = final_per_layer[layer_idx]

    # stack all hidden layer vectors
    all_stack = np.stack(
        [final_per_layer[i] for i in sorted(final_per_layer.keys())],
        axis=0
    )

    # average all hidden layers into one vector
    result["all_layers_mean"] = np.mean(
        all_stack,
        axis=0
    ).astype(np.float32)

    return result


# load models
def load_wav2vec2_sid_model(device="cpu"):
    # define the pretrained speaker model
    model_name = "superb/wav2vec2-base-superb-sid"

    # load the audio feature extractor
    feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(model_name)

    # load the wav2vec2 model with hidden states
    model = Wav2Vec2ForSequenceClassification.from_pretrained(
        model_name,
        output_hidden_states=True,
    )

    # prepare the model for inference
    model.eval()
    model.to(device)

    return feature_extractor, model


def load_hubert_sid_model(device="cpu"):
    # define the pretrained speaker model
    model_name = "superb/hubert-base-superb-sid"

    # load the audio feature extractor
    feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(model_name)

    # load the hubert model with hidden states
    model = HubertForSequenceClassification.from_pretrained(
        model_name,
        output_hidden_states=True,
    )

    # prepare the model for inference
    model.eval()
    model.to(device)

    return feature_extractor, model


# load wav2vec2
print("Loading Wav2Vec2...")
wav2vec2_feature_extractor, wav2vec2_model = load_wav2vec2_sid_model(device=device)
print("Wav2Vec2 loaded.")

# load hubert
print("Loading HuBERT...")
hubert_feature_extractor, hubert_model = load_hubert_sid_model(device=device)
print("HuBERT loaded.")


# find patient-only WAV files

# search recursively for patient audio files
patient_wavs = sorted(PREPROCESSED_ROOT.rglob("*_patient.wav"))

# show the number of files found
print("Found patient-only WAVs:", len(patient_wavs))
for p in patient_wavs[:10]:
    print(p)
# stop when no matching files exist
assert len(patient_wavs) > 0, "No *_patient.wav files found. Check preprocessing output."


# extract and save features

# store metadata for each recording
feature_rows = []

# choose the hidden layers and segment duration
SELECTED_LAYERS = (6, -1)
SEGMENT_SECONDS = 10

# process every patient recording
for wav_path in tqdm(patient_wavs):
    wav_path = Path(wav_path)

    # Expected:
    # preprocessed_patient_audio / DatasetName / wav_patient_only / file_patient.wav

    # identify the dataset and recording
    dataset_name = wav_path.parents[1].name
    file_id = wav_path.stem.replace("_patient", "")

    # create one output folder per dataset
    dataset_out_dir = FEATURE_ROOT / dataset_name
    dataset_out_dir.mkdir(parents=True, exist_ok=True)

    try:
        # -----------------------------
        # HuBERT
        # -----------------------------

        # extract hubert hidden layer features
        hubert_features = extract_ssl_selected_features(
            wav_path=wav_path,
            feature_extractor=hubert_feature_extractor,
            model=hubert_model,
            selected_layers=SELECTED_LAYERS,
            segment_seconds=SEGMENT_SECONDS,
            device=device,
        )

        # define hubert output paths
        hubert_last_path = dataset_out_dir / f"{file_id}_hubert_last.npy"
        hubert_layer6_path = dataset_out_dir / f"{file_id}_hubert_layer_6.npy"
        hubert_allmean_path = dataset_out_dir / f"{file_id}_hubert_all_layers_mean.npy"

        # save hubert vectors
        np.save(hubert_last_path, hubert_features["last"])
        np.save(hubert_layer6_path, hubert_features["layer_6"])
        np.save(hubert_allmean_path, hubert_features["all_layers_mean"])

        # -----------------------------
        # Wav2Vec2
        # -----------------------------

        # extract wav2vec2 hidden layer features
        wav2vec2_features = extract_ssl_selected_features(
            wav_path=wav_path,
            feature_extractor=wav2vec2_feature_extractor,
            model=wav2vec2_model,
            selected_layers=SELECTED_LAYERS,
            segment_seconds=SEGMENT_SECONDS,
            device=device,
        )

        # define wav2vec2 output paths
        wav2vec2_last_path = dataset_out_dir / f"{file_id}_wav2vec2_last.npy"
        wav2vec2_layer6_path = dataset_out_dir / f"{file_id}_wav2vec2_layer_6.npy"
        wav2vec2_allmean_path = dataset_out_dir / f"{file_id}_wav2vec2_all_layers_mean.npy"

        # save wav2vec2 vectors
        np.save(wav2vec2_last_path, wav2vec2_features["last"])
        np.save(wav2vec2_layer6_path, wav2vec2_features["layer_6"])
        np.save(wav2vec2_allmean_path, wav2vec2_features["all_layers_mean"])

        # record successful extraction metadata
        feature_rows.append({
            "dataset": dataset_name,
            "file_id": file_id,
            "wav_path": str(wav_path),

            "hubert_last_path": str(hubert_last_path),
            "hubert_layer6_path": str(hubert_layer6_path),
            "hubert_all_layers_mean_path": str(hubert_allmean_path),

            "wav2vec2_last_path": str(wav2vec2_last_path),
            "wav2vec2_layer6_path": str(wav2vec2_layer6_path),
            "wav2vec2_all_layers_mean_path": str(wav2vec2_allmean_path),

            "feature_shape": str(tuple(hubert_features["last"].shape)),
            "status": "ok",
            "error": "",
        })

    # record errors without stopping other files
    except Exception as e:
        feature_rows.append({
            "dataset": dataset_name,
            "file_id": file_id,
            "wav_path": str(wav_path),

            "hubert_last_path": "",
            "hubert_layer6_path": "",
            "hubert_all_layers_mean_path": "",

            "wav2vec2_last_path": "",
            "wav2vec2_layer6_path": "",
            "wav2vec2_all_layers_mean_path": "",

            "feature_shape": "",
            "status": "error",
            "error": str(e),
        })

    # release unused gpu memory
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # release unused python objects
    gc.collect()



# convert extraction records to a dataframe
ssl_metadata = pd.DataFrame(feature_rows)

# save paths statuses and shapes
metadata_path = FEATURE_ROOT / "ssl_feature_metadata.csv"
ssl_metadata.to_csv(metadata_path, index=False)
print("\nSaved metadata:", metadata_path)
print("Total:", len(ssl_metadata))
print("\nStatus counts:")
print(ssl_metadata["status"].value_counts())
print("\nBy dataset/status:")
print(ssl_metadata.groupby(["dataset", "status"]).size())


# Create combined matrices


# keep successfully extracted recordings
valid_df = ssl_metadata[ssl_metadata["status"] == "ok"].copy()

# stop when no valid features exist
assert len(valid_df) > 0, "No valid features extracted."

# map feature names to metadata path columns
FEATURE_TYPES = {
    "hubert_last": "hubert_last_path",
    "hubert_layer_6": "hubert_layer6_path",
    "hubert_all_layers_mean": "hubert_all_layers_mean_path",

    "wav2vec2_last": "wav2vec2_last_path",
    "wav2vec2_layer_6": "wav2vec2_layer6_path",
    "wav2vec2_all_layers_mean": "wav2vec2_all_layers_mean_path",
}


# create one npz matrix for each feature type
for feature_name, path_col in FEATURE_TYPES.items():

    # store vectors and aligned metadata
    X = []
    dataset = []
    file_id = []
    wav_paths = []
    feature_paths = []

    # load every valid feature vector
    for _, row in valid_df.iterrows():
        feat = np.load(row[path_col])

        # convert the feature to one flat vector
        feat = feat.reshape(-1).astype(np.float32)

        X.append(feat)
        dataset.append(row["dataset"])
        file_id.append(row["file_id"])
        wav_paths.append(row["wav_path"])
        feature_paths.append(row[path_col])

    # stack vectors into one matrix
    X = np.vstack(X).astype(np.float32)

    # define the output archive path
    out_npz = FEATURE_ROOT / f"{feature_name}_matrix.npz"

    # save the matrix and aligned metadata
    np.savez(
        out_npz,
        X=X,
        dataset=np.array(dataset),
        file_id=np.array(file_id),
        wav_path=np.array(wav_paths),
        feature_path=np.array(feature_paths),
    )
    print("\nSaved:", out_npz)
    print(feature_name, "X shape:", X.shape)



print("\nDone.")
print("Feature root:", FEATURE_ROOT)
print("Metadata:", metadata_path)
print("\nRecommended files for modeling:")
for feature_name in FEATURE_TYPES:
    print(FEATURE_ROOT / f"{feature_name}_matrix.npz")

